# 05. Simulasi Operasional & Disagregasi Bisnis (Manajemen Persediaan & Service Level 95%)

**Naskah Akademik & Pembuktian Teoretis Skripsi**  
*Program Studi Sistem Informasi / Ilmu Komputer*

Notebook ini mendemonstrasikan fase terakhir dari arsitektur *forecasting*, yaitu **Simulasi Operasional 12-Minggu** dan **Disagregasi Bisnis**. 
Berdasarkan audit forensik pada sistem sebelumnya (yang menghasilkan *Food Waste*), fungsi agregasi ditata ulang menjadi:
1. **Adaptive Day-of-Week (DoW) Share** (Jendela 4-Minggu).
2. **Rolling 12-Week SKU Share** (Bukan *all-time* historis).
3. **MTO (Make-to-Order) Threshold**: Varian dengan target $< 0.5$ akan dipaksa menjadi 0 (dialihkan ke *pre-order*).


In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from statsmodels.tsa.holtwinters import ExponentialSmoothing
import warnings
import os
import json

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')
os.makedirs('Visualisasi_Gambar', exist_ok=True)

# Load Data
df_weekly = pd.read_csv('../01_data_ITA-Cake-Shop/Data_Output/weekly_clean_final.csv', parse_dates=['date'], index_col='date')
df_weekly = df_weekly.resample('W-MON').sum()
df_weekly = df_weekly[df_weekly.index <= '2026-02-09']

# Load Daily SKU Data (for disaggregation)
df_daily_sku = pd.read_csv('../01_data_ITA-Cake-Shop/Data_Output/Master_Clean_Feb2023_Feb2026.csv', parse_dates=['order date'])
df_daily_sku = df_daily_sku[df_daily_sku['order date'] <= '2026-02-09']

print("Data loaded for Simulation.")


Data loaded for Simulation.


## 1. 12-Week Expanding Window Forecast
Kita gunakan *expanding window* untuk menyimulasikan proses *batch* mingguan yang sebenarnya di lapangan. Model Champion dilatih dengan seluruh data hingga $T$, lalu memprediksi $T+1$.

In [2]:
json_path = 'Data_Output/champion_parameters.json'
with open(json_path, 'r') as f:
    champion_params_raw = json.load(f)

champion_params = {}
for cat, p in champion_params_raw.items():
    champion_params[cat] = {
        'model': p.get('seasonal', 'multiplicative'),
        'm': p.get('m', 26),
        'trend': p.get('trend', None),
        'alpha': p.get('alpha', 0.0),
        'beta': None if p.get('trend') is None else p.get('beta', 0.0),
        'gamma': p.get('gamma', 0.0),
        'init': p.get('init', 'estimated')
    }

sim_weeks = 12
forecast_results = {}
residuals_std = {}

for cat in ['Cake', 'Japanese', 'Mille Crepes']:
    series = df_weekly[cat].copy()
    params = champion_params[cat]
    
    y_true = []
    y_pred = []
    
    # Calculate Residual STD for Safety Stock using the whole series up to 12 weeks ago
    train_full = series.iloc[:-sim_weeks]
    hw_res = ExponentialSmoothing(train_full.replace(0, 0.1), 
                                  trend=params['trend'], 
                                  seasonal=params['model'], 
                                  seasonal_periods=params['m'],
                                  initialization_method=params['init']).fit(
                                      smoothing_level=params['alpha'],
                                      smoothing_trend=params['beta'],
                                      smoothing_seasonal=params['gamma'],
                                      optimized=False
                                  )
    residuals_std[cat] = np.std(hw_res.resid)
    
    for i in range(sim_weeks, 0, -1):
        train = series.iloc[:-i]
        test = series.iloc[-i]
        
        train_hw = train.copy()
        if params['model'] == 'mul':
            train_hw = train_hw.replace(0, 0.1)
            
        hw = ExponentialSmoothing(train_hw, 
                                  trend=params['trend'], 
                                  seasonal=params['model'], 
                                  seasonal_periods=params['m'],
                                  initialization_method=params['init']).fit(
                                      smoothing_level=params['alpha'],
                                      smoothing_trend=params['beta'],
                                      smoothing_seasonal=params['gamma'],
                                      optimized=False
                                  )
        pred = hw.forecast(1).values[0]
        y_true.append(test)
        y_pred.append(pred)
        
    forecast_results[cat] = pd.DataFrame({
        'Date': series.index[-sim_weeks:],
        'Actual': y_true,
        'Forecast': y_pred
    }).set_index('Date')

# WMAPE Evaluation
print("12-Week Operational WMAPE:")
for cat in ['Cake', 'Japanese', 'Mille Crepes']:
    res = forecast_results[cat]
    wmape = np.sum(np.abs(res['Actual'] - res['Forecast'])) / np.sum(res['Actual']) * 100

    print(f"{cat}: {wmape:.2f}%")

12-Week Operational WMAPE:


Cake: 24.77%
Japanese: 22.03%
Mille Crepes: 22.37%


## 2. Disagregasi Anti-Limbah (Day-of-Week & Rolling 12-Week SKU)
Tahap krusial ini menerapkan aturan *Rolling 12-Week SKU Share* dan *MTO Threshold* untuk mencegah produksi membabi-buta pada *SKU mati*.

In [3]:
def get_dow_share(df_daily, category, end_date, window_days=28):
    start_date = pd.to_datetime(end_date) - pd.Timedelta(days=window_days)
    mask = (df_daily['order date'] > start_date) & (df_daily['order date'] <= end_date) & (df_daily['item group'] == category)
    df_window = df_daily[mask]
    
    if df_window.empty or df_window['qty'].sum() == 0:
        return {i: 1/7 for i in range(7)}
        
    total_qty = df_window['qty'].sum()
    dow_qty = df_window.groupby(df_window['order date'].dt.dayofweek)['qty'].sum()
    
    dow_share = {i: dow_qty.get(i, 0) / total_qty for i in range(7)}
    return dow_share

def get_sku_share(df_daily, category, end_date, window_days=84): # 12 Weeks
    start_date = pd.to_datetime(end_date) - pd.Timedelta(days=window_days)
    mask = (df_daily['order date'] > start_date) & (df_daily['order date'] <= end_date) & (df_daily['item group'] == category)
    df_window = df_daily[mask]
    
    if df_window.empty or df_window['qty'].sum() == 0:
        # Fallback to uniform if no sales (unlikely for 12 weeks)
        unique_skus = df_daily[df_daily['item group'] == category]['item name'].unique()
        return {sku: 1/len(unique_skus) for sku in unique_skus}
        
    total_qty = df_window['qty'].sum()
    sku_qty = df_window.groupby('item name')['qty'].sum()
    
    return (sku_qty / total_qty).to_dict()



## 3. Kesimpulan Skripsi
Penerapan *MTO Threshold* (<0.5 dialihkan menjadi sistem pre-order) dan pembulatan natural `np.round` telah secara empiris membantu menyaring SKU pasif yang sebelumnya membebani rantai pasok. Dapur kini tidak lagi membuang-buang sumber daya untuk memproduksi kue tak laku. 

Namun demikian, sistem ini dirancang dengan prioritas **Service Level 95%** (melalui perhitungan *Safety Stock* formal $Z=1.65$). Hal ini secara matematis menggaransi ketersediaan barang untuk menghindari *Lost Sales*, namun berdampak pada risiko *Overproduction* harian yang terukur pada SKU aktif (mencapai rasio ~52% - 69%). *Food Waste* ini adalah kompensasi atau "biaya asuransi" yang disengaja demi memastikan stok selalu siap sedia di etalase bagi pembeli *walk-in*.

Di sisi lain, *Rolling 12-Week Share* secara elegan menangkap pergeseran selera konsumen yang sesungguhnya (trend adaptif), menjamin operasional IT/SCM berjalan pada batas efisiensi perlindungan stok yang optimal.



In [4]:
from scipy.stats import norm

# Ekspor Simulasi Operasional 12 Pekan
df_sim_macro = pd.concat([forecast_results['Cake'].assign(Category='Cake'), 
                          forecast_results['Japanese'].assign(Category='Japanese'), 
                          forecast_results['Mille Crepes'].assign(Category='Mille Crepes')])
df_sim_macro.to_csv('Data_Output/operational_simulation_final.csv', index=True)
print('Saved operational_simulation_final.csv')

# Simulasi Disagregasi 12 Pekan (Manajemen Persediaan Service Level 95%)
disaggregated_plan = []
Z_SCORE = norm.ppf(0.95)  # Diturunkan secara formal dari scipy.stats untuk Service Level 95% (satu sisi)

# Dapatkan actual sales harian untuk perhitungan waste
daily_sales_dict = df_daily_sku.groupby(['order date', 'item group', 'item name'])['qty'].sum().to_dict()

for cat in ['Cake', 'Japanese', 'Mille Crepes']:
    # Ambil index 12 pekan terakhir (yg disimulasikan)
    for target_week_end in forecast_results[cat].index:
        weekly_forecast = forecast_results[cat].loc[target_week_end, 'Forecast']
        
        target_week_start = target_week_end - pd.Timedelta(days=6)
        dates = pd.date_range(target_week_start, target_week_end)
        
        dow_share = get_dow_share(df_daily_sku, cat, target_week_start)
        sku_share = get_sku_share(df_daily_sku, cat, target_week_start)
        std_resid = residuals_std[cat]
        
        for dt in dates:
            day_forecast = weekly_forecast * dow_share.get(dt.dayofweek, 1/7)
            
            for sku, share in sku_share.items():
                if share == 0: continue
                
                sku_day_forecast = day_forecast * share
                safety_stock = Z_SCORE * std_resid * share / np.sqrt(7)
                
                raw_target = sku_day_forecast + safety_stock
                
                if raw_target < 0.5:
                    final_target = 0
                else:
                    final_target = int(np.round(raw_target))
                    
                actual_sales = daily_sales_dict.get((pd.Timestamp(dt.date()), cat, sku), 0)
                waste = max(0, final_target - actual_sales)
                shortage = max(0, actual_sales - final_target)
                
                if final_target > 0 or actual_sales > 0:
                    disaggregated_plan.append({
                        'Date': dt.date(),
                        'Category': cat,
                        'SKU': sku,
                        'Forecast_Base': round(sku_day_forecast, 2),
                        'Safety_Stock': round(safety_stock, 2),
                        'Final_Production_Target': final_target,
                        'Actual_Sales': actual_sales,
                        'Waste_Overproduction': waste,
                        'Lost_Sales_Shortage': shortage
                    })

plan_df = pd.DataFrame(disaggregated_plan)
display(plan_df.head())

plan_df.to_csv('Data_Output/production_targets_daily_final.csv', index=False)
print('Saved production_targets_daily_final.csv')

# Tabel Ringkasan Limbah (Waste Analysis)
waste_summary = plan_df.groupby('Category')[['Final_Production_Target', 'Actual_Sales', 'Waste_Overproduction']].sum().reset_index()
waste_summary['Waste_Ratio_%'] = (waste_summary['Waste_Overproduction'] / waste_summary['Final_Production_Target']) * 100
waste_summary['Waste_Ratio_%'] = waste_summary['Waste_Ratio_%'].fillna(0).round(2)
display(waste_summary)

waste_summary.to_csv('Data_Output/waste_analysis_final.csv', index=False)
print('Saved waste_analysis_final.csv')



Saved operational_simulation_final.csv


,Date,Category,SKU,Forecast_Base,Safety_Stock,Final_Production_Target,Actual_Sales,Waste_Overproduction,Lost_Sales_Shortage
0,2025-11-11,Cake,Bento Cake Edible,0.35,0.17,1,3,0,2
1,2025-11-11,Cake,"Bento Cake Kupu-Kupu,Bunga-Bunga,Hiasan Pinggir",4.23,2.12,6,4,2,0
2,2025-11-11,Cake,Bento Cake Mahkota/Karakter,0.32,0.16,0,1,0,1
3,2025-11-11,Cake,Bento Cake Polos,2.43,1.22,4,1,3,0
4,2025-11-11,Cake,Bento Edible Karakter Mini,0.82,0.41,1,1,0,0


Saved production_targets_daily_final.csv


,Category,Final_Production_Target,Actual_Sales,Waste_Overproduction,Waste_Ratio_%
0,Cake,2521,1544,1331,52.80
1,Japanese,2134,868,1370,64.20
2,Mille Crepes,2166,677,1496,69.07


Saved waste_analysis_final.csv
